<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/devoirs/Homework_Medical_QA_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 1 [rattrapage]: Classification de Réponses Médicales en Français

**Cours:** Informatique et Intelligence Artificielle

**Enseignant:** Emmanuel Noutahi, PhD

**Note:** /20 points

---

## Objectif

Développer un modèle de classification de texte médical pour identifier si une réponse à une question médicale est **correcte** ou **incorrecte**. Vous utiliserez un modèle de language français (**CamemBERTa v2** - almanach/camembertav2-base) pour cette tâche de classification binaire.

## Barème (/20)

1. Chargement et transformation des données (4 pts)
2. Analyse exploratoire (2 pts)
3. Préparation du modèle (3 pts)
4. Entraînement (3 pts)
5. Évaluation (3 pts)
6. Test sur nouveaux exemples (2 pts)
7. Questions de réflexion (3 pts)

**Ressources:**
- Dataset MediQAl: https://huggingface.co/datasets/ANR-MALADES/MediQAl
- Transformers: https://huggingface.co/docs/transformers

## Instructions

- Travail individuel, en binôme ou en trinôme (max 3 personnes)
- Les sections à compléter sont marquées `# TODO`
- Avant remise : **Tout exécuter**, vérifier les erreurs, mettre vos noms



### ÉQUIPE

- **<font color='red'>Nom Prénoms</font>**

## SECTION 1: Installation et Imports

**✅ Pas de modifications nécessaires - exécutez simplement les cellules**

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
plt.style.use("default")
sns.set_palette("husl")

print("✓ Bibliothèques importées avec succès")

In [ ]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositif utilisé: {device}")

## SECTION 2: Chargement et Transformation des Données (4 pts)

Le dataset **MediQAl** contient des questions médicales en français au format QCM (questions à choix multiples).

**Objectif:** Transformer le format QCM en classification binaire :
- Pour chaque question, créer UN exemple en échantillonnant soit la **bonne** réponse (label=1), soit une **mauvaise** réponse (label=0)
- Le texte final : `Cas clinique (si présent) + Question + Réponse proposée`

In [ ]:
print("Chargement du dataset MediQAl...")
dataset = load_dataset("ANR-MALADES/MediQAl", "mcqu")

# Sous-ensemble pour l'entraînement rapide
N_TRAIN = 2000
N_VAL = 500
N_TEST = 500

train_sample = dataset["train"].shuffle(seed=42).select(range(N_TRAIN))
val_sample = dataset["validation"].shuffle(seed=42).select(range(N_VAL))
test_sample = dataset["test"].shuffle(seed=42).select(range(N_TEST))

print(
    f"✓ Échantillons: train={len(train_sample)}, val={len(val_sample)}, test={len(test_sample)}"
)

### Question 2.1 (3 pts): Implémenter la transformation binaire

Complétez les fonctions `transform_to_binary` et `create_binary_dataset`.

**Structure des exemples MediQAl (config mcqu):** `clinical_case`, `question`, `answer_a` à `answer_e`, `correct_answers` (lettre, ex: 'A'), `medical_subject`

**`transform_to_binary(example, seed=42)`** doit retourner un dictionnaire avec :
- `text`: contexte + question + "Réponse proposée: ..."
- `label`: 1 (correcte) ou 0 (incorrecte) - 50% chacun
- `specialty`: `example['medical_subject']`

**`create_binary_dataset(original_data, seed=42)`** applique `transform_to_binary` à chaque exemple et retourne un DataFrame pandas.

In [ ]:
# ===========================
# TODO: Complétez transform_to_binary et create_binary_dataset

def transform_to_binary(example, seed=42):
    """Transforme un QCM en 1 exemple binaire (50% positif, 50% négatif)."""
    np.random.seed(seed + int(example['id']))

    context = ""
    if example["clinical_case"]:
        context = f"Cas clinique: {example['clinical_case']}\n\n"
    context += f"Question: {example['question']}"

    all_answers = {
        "A": example["answer_a"],
        "B": example["answer_b"],
        "C": example["answer_c"],
        "D": example["answer_d"],
        "E": example["answer_e"],
    }

    # TODO: Identifiez la lettre correcte et la réponse correcte
    correct_letter = ...
    correct_answer = ...
    incorrect_letters = [k for k in all_answers.keys() if k != correct_letter]
    wrong_letter = np.random.choice(incorrect_letters)
    incorrect_answer = ...


    # TODO: Avec probabilité 0.5, retournez un exemple positif, sinon négatif
    # Completez l'example negatif en vous inspirant du cas positif
    if np.random.random() < 0.5:
        # Positive example (correct answer)
        return {
            "text": f"{context}\n\nRéponse proposée: {correct_answer}",
            "label": 1,
            "specialty": example["medical_subject"],
        }
    else:
        # Negative example (random incorrect answer)
        return {
            "text": ...,
            "label": ...,
            "specialty": example["medical_subject"],
        }
        
def create_binary_dataset(original_data, seed=42):
    """Applique transform_to_binary à chaque exemple et retourne un DataFrame."""
    all_examples = []
    for example in original_data:
        # TODO: appelez transform_to_binary et ajoutez le résultat à all_examples
        binary_example = ...
        all_examples.append(binary_example)
    return pd.DataFrame(all_examples)
# ===========================

### Question 2.2 (1 pt)
Montrez le premier exemple positif et le premier exemple négatif du dataset

In [ ]:
df_train = create_binary_dataset(train_sample)

In [ ]:
# Exemple transformé
# TODO: accédez et afficher le premier exemple positive (reéponse correct) de df_train
# utilisez la colonne text
print("=" * 80)
print("EXEMPLE POSITIF (Label=1, Bonne réponse)")
print("=" * 80)
pos_ex = ...
print(pos_ex["text"])

print("=" * 80)
print("EXEMPLE NÉGATIF (Label=0, Mauvaise réponse)")
print("=" * 80)
# TODO: accédez et afficher le premier exemple négative (reéponse incorrect) de df_train
# utilisez la colonne text
neg_ex = ...
print(neg_ex["text"])

## SECTION 3: Analyse Exploratoire (2 pts)

### Question 3.1 (2 pts): Distribution des spécialités

Créez un **countplot** montrant le nombre d'exemples par spécialité médicale, coloré par le label (0 ou 1). Utilisez `sns.countplot` avec `data=df_train`, `x='specialty'`, `hue='label'`.

In [ ]:
# ===========================
# TODO: Créez le countplot des spécialités par label sur une figure de taille (12, 8)
fig = ...
ax = sns.countplot(data=..., x=..., hue=...)
ax.set_ylabel("Nombre d'exemples", fontsize=12)
ax.set_title("Spécialités Médicales", fontsize=14, fontweight='bold')
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()
# ===========================

## SECTION 4: Préparation du Modèle (3 pts)

1. Charger le tokenizer **CamemBERTa v2** (almanach/camembertav2-base)
2. Créer les jeux val et test avec `create_binary_dataset`
3. Tokeniser les textes et préparer les datasets pour le Trainer

### Question 4.1 (1 pt): Charger le tokenizer

Chargez le tokenizer pour `almanach/camembertav2-base` avec `AutoTokenizer.from_pretrained()`.

In [ ]:
# TODO: Chargez le tokenizer
model_name = "almanach/camembertav2-base"
tokenizer = ...
print(f"✓ Tokenizer chargé: {model_name}")

### Question 4.2 (1 pt): Créer les datasets val et test

Appliquez `create_binary_dataset` à `val_sample` et `test_sample`, puis créez `hf_train`, `hf_val`, `hf_test` avec `Dataset.from_pandas` en sélectionnant les colonnes `text` et `label`.

In [ ]:
# TODO: Créez df_val, df_test,, hf_val, hf_test
hf_train = Dataset.from_pandas(df_train[["text", "label"]])

df_val = ...
hf_val = ...

df_test = ...
hf_test = ...

print(f"hf_train: {len(hf_train)}, hf_val: {len(hf_val)}, hf_test: {len(hf_test)}")

### Question 4.3 (1 pts): Tokenisation et Métriques

Exécutez la cellule : la fonction `tokenize_function` tokenise avec `padding='max_length'`, `truncation=True`, `max_length=512`. Les datasets sont préparés pour le Trainer.

In [ ]:
# Conversion en Datasets HuggingFace
# RIEN à changer ici
def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=512
    )


print("Tokenization...")
tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_val = hf_val.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

# Préparation pour PyTorch
tokenized_train = tokenized_train.remove_columns(["text"]).rename_column(
    "label", "labels"
)
tokenized_val = tokenized_val.remove_columns(["text"]).rename_column("label", "labels")
tokenized_test = tokenized_test.remove_columns(["text"]).rename_column(
    "label", "labels"
)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

print("✓ Datasets prêts pour l'entraînement")

In [ ]:
# TODO: Complétez compute_metrics avec 
# accuracy, f1, precision, recall (average='binary')
# vous devez mettre en arguments les variables predictions et labels dans le bon ordre
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': ...,
        'f1': f1_score(..., average='binary'),
        'precision': precision_score(..., average='binary'),
        'recall': recall_score(...,  average='binary')
    }

## SECTION 5: Entraînement (3 pts)

### Question 5.1 (3 pts): Initialiser le modèle et lancer l'entraînement

1. Chargez le modèle avec `AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)`
2. Créez `TrainingArguments` avec : 
   1. pour 2 epochs 
   2. learning rate de 5e-5
   3. logging chaque 50 steps
   4. evaluation chaque 50 steps
3. Créez le `Trainer` et appelez `trainer.train()` pour entrainer le modèle pour 2 epochs

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


# TODO: Créez TrainingArguments et Trainer
# Ajouter les arguments manquants en remplacant les ...
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=...,
    logging_steps=...,
    learning_rate=...,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=12,
    num_train_epochs=...,
    weight_decay=0.0001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Début de l'entraînement...")
train_result = trainer.train()
print(f"✓ Entraînement terminé en {train_result.metrics['train_runtime']:.2f}s")

## SECTION 6: Évaluation (3 pts)

### Question 6.1 (2 pts): Métriques sur le test

Évaluez le modèle sur `tokenized_test` avec `trainer.evaluate()` et affichez accuracy, F1, precision, recall.

In [ ]:
# TODO: Évaluez sur le jeu de test et affichez les métriques
trained_results = ...
print(f"Accuracy:  {trained_results['eval_accuracy']:.4f}")
print(f"F1-Score:  {trained_results['eval_f1']:.4f}")
print(f"Precision: ....")
print(f"Recall:    ....")

Obtenez les prédictions avec `trainer.predict(tokenized_test)`, puis tracez la matrice de confusion avec `ConfusionMatrixDisplay`. Labels d'affichage : `['Incorrect', 'Correct']`.

In [ ]:
# Pas de code à changer ici
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Incorrect", "Correct"]
)
disp.plot(cmap="Blues", ax=ax, values_format="d")
ax.set_title("Matrice de Confusion - Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("\nDétails:")
print(f"  Vrais Négatifs (TN):  {tn} - Réponses incorrectes correctement identifiées")
print(f"  Faux Positifs (FP):   {fp} - Réponses incorrectes classées comme correctes ⚠️")
print(f"  Faux Négatifs (FN):   {fn} - Réponses correctes classées comme incorrectes ⚠️")
print(f"  Vrais Positifs (TP):  {tp} - Réponses correctes correctement identifiées")

### Question 6.2 (1 pts): Erreur du modèle

**QUESTION: QUE REMARQUEZ VOUS SUR LA MATRICE DE CONFUSION ? Quel est le type d'erreur le plus fréquent que le modèle fait ?**

**<font color='red'>Réponse à la question sur ce que vous remarquez
</font>**

## SECTION 7: Test sur Nouveaux Exemples (2 pts)

### Question 7.1 (2 pts): Fonction de prédiction

Implémentez `predict_answer(question, answer, clinical_case=None)` qui :
1. Formate le texte comme à l'entraînement (cas clinique + question + réponse)
2. Tokenise avec `tokenizer(text, return_tensors='pt', truncation=True, max_length=512)`
3. Passe sur device, évalue le modèle, retourne `(prediction, confidence)` où `prediction` est 0 ou 1 et `confidence` est la proba softmax

In [ ]:
# TODO: Complétez predict_answer
def predict_answer(question, answer, clinical_case=None):
    text = ""
    if clinical_case:
        text = f"Cas clinique: {clinical_case}\n\n"
    text += f"Question: {question}\n\nRéponse proposée: {answer}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = inputs.to(device)
    model.eval()
    with torch.no_grad():
        # TODO: generez les outputs du modèle en fonction des inputs
        outputs = ...
        probs = torch.softmax(outputs.logits, dim=1)
        # TODO: prediction = torch.argmax des logits (0 ou 1), confidence = proba du bloc prédit
        prediction = ...
        confidence = confidence = probs[0][prediction].item()
    return prediction, confidence

# Test
q = "Patient avec douleur thoracique irradiant au bras gauche. Diagnostic ?"
pred_c, conf_c = predict_answer(q, "Infarctus du myocarde")
pred_i, conf_i = predict_answer(q, "Pneumonie")
print(f"Bonne réponse: {'✓' if pred_c == 1 else '✗'} (confiance: {conf_c:.1%})")
print(f"Mauvaise réponse: {'✗' if pred_i == 0 else '✓'} (confiance: {conf_i:.1%})")

## SECTION 8: Questions de Réflexion (3 pts)

### Question 8.1 (1 pt): Métriques en contexte médical

Comparez accuracy, precision et recall. Quelle métrique est la plus importante en contexte médical pour ce type d'application ? Justifiez en quelques lignes.

**Votre réponse:**

### Question 8.2 (2 pt): Utilisation clinique

Le modèle est-il prêt pour une utilisation clinique directe ? Donnez 2 limitations et proposez 1 scénario d'usage réaliste (avec supervision humaine).

**Votre réponse:**

## Remise

1. Exécution → Tout exécuter
2. Vérifier les erreurs
3. Compléter les réponses textuelles
4. Sauvegarder et partager le lien

**Bon travail !**